# WFP ADAM — Historical National Exposure

Retrieves country-level population exposure for all tropical storms, using the WFP ADAM OGC API Features endpoint for full historical coverage. For each storm, retrieves only the latest available episode. Saves national level exposure estimates to blob storage.

**API path:**
```
collections/adam.adam_ts_events/items  (paginated)
  -> deduplicate to latest episode_id per event_id
    -> population_csv_url -> aggregate ADM2 rows by ADM0_NAME
```

**Wind speed thresholds in output columns:**
- `pop_60kmh` — population exposed to winds ≥ 60 km/h (~32 kt, tropical storm force)
- `pop_90kmh` — population exposed to winds ≥ 90 km/h (~49 kt)
- `pop_120kmh` — population exposed to winds ≥ 120 km/h (~65 kt, hurricane force)

In [10]:
import base64
import json
import requests
import pandas as pd
import pycountry
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()

ADAM_BASE  = 'https://api.adam.geospatial.wfp.org/api'
COLLECTION = 'adam.adam_ts_events'
PAGE_SIZE  = 100
OUTPUT_CSV = 'adam_historical_national_exposure.csv'

## 1. Fetch all TC event records

In [11]:
all_items = []
offset    = 0

while True:
    resp = requests.get(
        f'{ADAM_BASE}/collections/{COLLECTION}/items',
        params={'limit': PAGE_SIZE, 'offset': offset},
        timeout=30,
    )
    resp.raise_for_status()
    # API returns base64-encoded JSON; decode before parsing
    if not resp.content:
        break
    data     = json.loads(base64.b64decode(resp.content))
    features = data.get('features', [])
    if not features:
        break

    for f in features:
        p = f.get('properties', f)   # GeoJSON features nest fields under 'properties'
        all_items.append({
            'event_id':          p.get('event_id'),
            'episode_id':        p.get('episode_id'),
            'uid':               p.get('uid', ''),
            'storm_name':        p.get('name', ''),
            'source':            p.get('source', ''),
            'from_date':         p.get('from_date', ''),
            'alert_level':       p.get('alert_level', ''),
            'population_csv_url': p.get('population_csv_url', ''),
        })

    print(f'Offset {offset:>5}: {len(features)} records fetched  (total so far: {len(all_items)})')
    offset += PAGE_SIZE
    if len(features) < PAGE_SIZE:
        break

df_all = pd.DataFrame(all_items)
print(f'\nTotal records fetched: {len(df_all)}')

Offset     0: 100 records fetched  (total so far: 100)
Offset   100: 100 records fetched  (total so far: 200)
Offset   200: 100 records fetched  (total so far: 300)
Offset   300: 100 records fetched  (total so far: 400)
Offset   400: 100 records fetched  (total so far: 500)
Offset   500: 100 records fetched  (total so far: 600)
Offset   600: 100 records fetched  (total so far: 700)
Offset   700: 100 records fetched  (total so far: 800)
Offset   800: 100 records fetched  (total so far: 900)
Offset   900: 100 records fetched  (total so far: 1000)
Offset  1000: 100 records fetched  (total so far: 1100)
Offset  1100: 100 records fetched  (total so far: 1200)
Offset  1200: 100 records fetched  (total so far: 1300)
Offset  1300: 100 records fetched  (total so far: 1400)
Offset  1400: 100 records fetched  (total so far: 1500)
Offset  1500: 100 records fetched  (total so far: 1600)
Offset  1600: 100 records fetched  (total so far: 1700)
Offset  1700: 100 records fetched  (total so far: 1800)
O

In [16]:
df_all

,event_id,episode_id,uid,storm_name,source,from_date,alert_level,population_csv_url
0,1000776,6,1000776_6,TWENTYSEVE-21,JTWC,2021-04-04T18:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2021...
1,1000775,8,1000775_8,SEROJA-21,JTWC,2021-04-04T12:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2021...
2,1000776,9,1000776_9,TWENTYSEVE-21,JTWC,2021-04-04T18:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2021...
3,1000775,11,1000775_11,SEROJA-21,JTWC,2021-04-04T12:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2021...
4,1000776,10,1000776_10,TWENTYSEVE-21,JTWC,2021-04-04T18:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2021...
...,...,...,...,...,...,...,...,...
8452,1001262,1,1001262_1,TWENTYSIX-26,JTWC,2026-03-06T00:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2026...
8453,1001262,2,1001262_2,TWENTYSIX-26,JTWC,2026-03-06T00:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2026...
8454,1001262,3,1001262_3,TWENTYSIX-26,JTWC,2026-03-06T00:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2026...
8455,1001262,4,1001262_4,TWENTYSIX-26,JTWC,2026-03-06T00:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2026...


## 2. Deduplicate to latest episode per event

In [17]:
# Keep only the row with the highest episode_id for each event_id
df_latest = (
    df_all
    .sort_values('episode_id', ascending=False)
    .drop_duplicates(subset='event_id', keep='first')
    .reset_index(drop=True)
)

print(f'Unique events (latest episode only): {len(df_latest)}')
df_latest.head()

Unique events (latest episode only): 818


,event_id,episode_id,uid,storm_name,source,from_date,alert_level,population_csv_url
0,1000510,70,1000510_70,LESLIE-18,NOAA,2018-09-23T15:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2018...
1,1000495,68,1000495_68,FLORENCE-18,NOAA,2018-08-31T21:00:00,Green,https://static.gis.wfp.org/adam_ts/events/2018...
2,1000588,64,1000588_64,DORIAN-19,NOAA,2019-08-24T15:00:00,Orange,https://static.gis.wfp.org/adam_ts/events/2019...
3,1000961,60,1000961_60,FREDDY-23,JTWC,2023-02-06T06:00:00,Red,https://static.gis.wfp.org/adam_ts/events/2023...
4,1000986,59,1000986_59,KHANUN-23,JTWC,2023-07-27T06:00:00,Orange,https://static.gis.wfp.org/adam_ts/events/2023...


## 3. For each event, fetch and aggregate population exposure

The `population_csv_url` contains ADM2-level rows. We aggregate by `ADM0_NAME` to get national totals, then map country names to ISO3 codes via `pycountry`.

> **Note:** Older records may return HTTP 403 for the CSV URL — these are skipped gracefully.

In [18]:
_iso3_cache = {}

def name_to_iso3(country_name):
    """Map a country name string to an ISO 3166-1 alpha-3 code via pycountry fuzzy search."""
    if country_name in _iso3_cache:
        return _iso3_cache[country_name]
    try:
        code = pycountry.countries.search_fuzzy(country_name)[0].alpha_3
    except LookupError:
        code = None
    _iso3_cache[country_name] = code
    return code


def fetch_national_exposure(row):
    """Fetch the population CSV for one event and return national-level rows."""
    csv_url = row['population_csv_url']
    if not csv_url:
        return []

    try:
        resp = requests.get(csv_url, timeout=30)
        resp.raise_for_status()
        df_csv = pd.read_csv(pd.io.common.StringIO(resp.text))
    except Exception as e:
        print(f'  [{row["event_id"]}] CSV fetch failed: {e}')
        return []

    # Normalise column names (strip whitespace, upper-case)
    df_csv.columns = df_csv.columns.str.strip().str.upper()

    pop_cols = ['POP_60_KMH', 'POP_90_KMH', 'POP_120_KMH']
    present  = [c for c in pop_cols if c in df_csv.columns]
    if 'ADM0_NAME' not in df_csv.columns or not present:
        return []

    agg = (
        df_csv
        .groupby('ADM0_NAME', dropna=False)[present]
        .sum()
        .reset_index()
    )

    out = []
    for _, r in agg.iterrows():
        adm0 = r['ADM0_NAME']
        rec  = {
            'event_id':    row['event_id'],
            'episode_id':  row['episode_id'],
            'storm_name':  row['storm_name'],
            'source':      row['source'],
            'from_date':   row['from_date'],
            'alert_level': row['alert_level'],
            'iso3':        name_to_iso3(str(adm0)),
            'country_name': adm0,
        }
        for col in pop_cols:
            out_col = col.lower().replace('_kmh', 'kmh')  # pop_60kmh etc.
            rec[out_col] = int(r[col]) if col in r and pd.notna(r[col]) else None
        out.append(rec)
    return out

In [19]:
all_rows = []

for _, ev in df_latest.iterrows():
    print(f'Fetching {ev["storm_name"]} ({ev["event_id"]}, ep {ev["episode_id"]}) …', end=' ')
    rows = fetch_national_exposure(ev)
    all_rows.extend(rows)
    print(f'{len(rows)} countries')

df = (
    pd.DataFrame(all_rows)
    .sort_values(['from_date', 'storm_name'], ascending=False)
    .reset_index(drop=True)
)

# Reorder columns
leading  = ['storm_name', 'event_id', 'episode_id', 'source', 'from_date',
            'alert_level', 'iso3', 'country_name']
pop_cols = sorted([c for c in df.columns if c.startswith('pop_')])
df = df[leading + pop_cols]

Fetching LESLIE-18 (1000510, ep 70) …   [1000510] CSV fetch failed: 403 Client Error: Forbidden for url: https://static.gis.wfp.org/adam_ts/events/2018/10/1000510_70/ADAM_TS_1000510_70_pop_estimation.csv
0 countries
Fetching FLORENCE-18 (1000495, ep 68) …   [1000495] CSV fetch failed: 403 Client Error: Forbidden for url: https://static.gis.wfp.org/adam_ts/events/2018/09/1000495_68/ADAM_TS_1000495_68_pop_estimation.csv
0 countries
Fetching DORIAN-19 (1000588, ep 64) …   [1000588] CSV fetch failed: 403 Client Error: Forbidden for url: https://static.gis.wfp.org/adam_ts/events/2019/09/1000588_64/ADAM_TS_1000588_64_pop_estimation.csv
0 countries
Fetching FREDDY-23 (1000961, ep 60) …   [1000961] CSV fetch failed: 403 Client Error: Forbidden for url: https://static.gis.wfp.org/adam_ts/events/2023/03/1000961_60/ADAM_TS_1000961_60_pop_estimation.csv
0 countries
Fetching KHANUN-23 (1000986, ep 59) … 6 countries
Fetching HECTOR-18 (1000477, ep 59) …   [1000477] CSV fetch failed: 403 Client Error

## 4. Clean results and join with IBTrACS

In [27]:
df_sel = df[df.source == "NOAA"]

Get the IBTrACS data.

In [30]:
engine = stratus.get_engine("prod")
with engine.connect() as conn:
    df_ibtracs = pd.read_sql("SELECT * FROM storms.ibtracs_storms", conn)

Prepare the ADAM data to join, per-storm, with IBTrACS.

In [70]:
def parse_name(input_name):
    season_str = input_name.split("-")[1]
    season = int("20" + season_str)
    storm_name = input_name.split("-")[0].split(" ")[-1]
    return season, storm_name

df_cleaned = df_sel.copy()

df_cleaned[['season', 'name']] = df_cleaned["storm_name"].apply(
    lambda x: pd.Series(parse_name(x))
)

df_sel_ = df_cleaned.drop_duplicates(subset=["storm_name"]).drop(columns=["iso3", "country_name", "pop_90kmh", "pop_120kmh", "pop_60kmh"]).reset_index()
print(f"Dataset has {len(df_sel_)} unique storms.")

Dataset has 47 unique storms.


Join, per-storm, with the IBTrACS records.

In [71]:
df_merged = df_sel_.merge(df_ibtracs, on=["season", "name"], how="left")
df_merged = df_merged.sort_values("provisional").drop_duplicates(subset="storm_name", keep="first")
assert(len(df_merged) == len(df_sel_))

Now join back to the exposure records.

In [72]:
df_final_merged = df_sel.merge(df_merged)
assert(len(df_final_merged) == len(df_sel))

## 5. Save output results to Azure

In [68]:
stratus.upload_csv_to_blob(df_final_merged, f'ds-cyclone-exposure/{OUTPUT_CSV}')